# Train Draft Model (80M) via Knowledge Distillation

**Teacher**: Qwen2.5-0.5B (494M params)  
**Student**: Draft model (80M params, 6 layers, dim=384)  
**Goal**: >30% acceptance rate + >100 tok/s on ANE

**50k steps on Blackwell ≈ 40-60 min**

In [ ]:
!pip install -q torch transformers datasets safetensors sentencepiece

In [ ]:
import math, os, time, json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from dataclasses import dataclass

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")

## 1. Define draft model (80M params)

In [ ]:
DRAFT_CONFIG = {
    "vocab_size": 151936,
    "hidden_size": 384,
    "num_attention_heads": 6,
    "num_key_value_heads": 2,
    "intermediate_size": 1536,
    "num_hidden_layers": 6,
    "rms_norm_eps": 1e-6,
    "rope_theta": 1000000.0,
    "max_seq_len": 128,
    "tie_word_embeddings": True,
}

@dataclass
class Cfg:
    vocab_size: int = 151936
    hidden_size: int = 384
    num_attention_heads: int = 6
    num_key_value_heads: int = 2
    intermediate_size: int = 1536
    num_hidden_layers: int = 6
    rms_norm_eps: float = 1e-6
    rope_theta: float = 1000000.0
    max_seq_len: int = 128
    tie_word_embeddings: bool = True


def rope(hd, sl, theta):
    f = 1.0 / (theta ** (torch.arange(0, hd, 2).float() / hd))
    e = torch.cat([torch.outer(torch.arange(sl).float(), f)] * 2, -1)
    return e.cos(), e.sin()

def rot(x):
    d = x.shape[-1] // 2
    return torch.cat([-x[..., d:], x[..., :d]], -1)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return self.w * (x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps))


class Attn(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.nh, self.nkv = c.num_attention_heads, c.num_key_value_heads
        self.hd = c.hidden_size // c.num_attention_heads
        self.ng = self.nh // self.nkv
        self.q = nn.Linear(c.hidden_size, self.nh * self.hd, bias=True)
        self.k = nn.Linear(c.hidden_size, self.nkv * self.hd, bias=True)
        self.v = nn.Linear(c.hidden_size, self.nkv * self.hd, bias=True)
        self.o = nn.Linear(self.nh * self.hd, c.hidden_size, bias=False)

    def forward(self, x, cos, sin, mask):
        B, L, _ = x.shape
        q = self.q(x).view(B, L, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, L, self.nkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, L, self.nkv, self.hd).transpose(1, 2)
        q = q * cos + rot(q) * sin
        k = k * cos + rot(k) * sin
        if self.ng > 1:
            k = k.unsqueeze(2).expand(B, self.nkv, self.ng, L, self.hd).reshape(B, self.nh, L, self.hd)
            v = v.unsqueeze(2).expand(B, self.nkv, self.ng, L, self.hd).reshape(B, self.nh, L, self.hd)
        a = F.softmax(torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(self.hd) + mask, dim=-1)
        return self.o(torch.matmul(a, v).transpose(1,2).contiguous().view(B, L, -1))


class FFN(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.g = nn.Linear(c.hidden_size, c.intermediate_size, bias=False)
        self.u = nn.Linear(c.hidden_size, c.intermediate_size, bias=False)
        self.d = nn.Linear(c.intermediate_size, c.hidden_size, bias=False)
    def forward(self, x):
        return self.d(F.silu(self.g(x)) * self.u(x))


class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.attn = Attn(c); self.ffn = FFN(c)
        self.n1 = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.n2 = RMSNorm(c.hidden_size, c.rms_norm_eps)
    def forward(self, x, cos, sin, mask):
        x = x + self.attn(self.n1(x), cos, sin, mask)
        return x + self.ffn(self.n2(x))


class Draft(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.cfg = c
        self.emb = nn.Embedding(c.vocab_size, c.hidden_size)
        self.layers = nn.ModuleList([Block(c) for _ in range(c.num_hidden_layers)])
        self.norm = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.head = nn.Linear(c.hidden_size, c.vocab_size, bias=False)
        if c.tie_word_embeddings:
            self.head.weight = self.emb.weight
        hd = c.hidden_size // c.num_attention_heads
        co, si = rope(hd, c.max_seq_len, c.rope_theta)
        self.register_buffer('rc', co.unsqueeze(0).unsqueeze(0))
        self.register_buffer('rs', si.unsqueeze(0).unsqueeze(0))
        m = torch.triu(torch.full((c.max_seq_len, c.max_seq_len), -1e9), diagonal=1)
        self.register_buffer('mask', m.unsqueeze(0).unsqueeze(0))

    def forward(self, ids):
        h = self.emb(ids)
        for layer in self.layers:
            h = layer(h, self.rc, self.rs, self.mask)
        return self.head(self.norm(h))

    def np(self): return sum(p.numel() for p in self.parameters())


c = Cfg(**DRAFT_CONFIG)
student = Draft(c).to(device)
n = student.np()
emb_n = c.vocab_size * c.hidden_size
layer_n = n - emb_n - c.hidden_size  # exclude norm
print(f"Student: {n:,} params")
print(f"  Embedding: {emb_n:,} ({emb_n/n:.0%})")
print(f"  Layers:    {layer_n:,} ({layer_n/n:.0%})")

## 2. Load teacher (Qwen2.5-0.5B)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

teacher = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B", torch_dtype=torch.float32
).to(device).eval()

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
print(f"Teacher: {sum(p.numel() for p in teacher.parameters()):,} params")
print(f"Compression: {sum(p.numel() for p in teacher.parameters()) / student.np():.1f}x")

## 3. Dataset (wikitext-103, 20M tokens)

In [ ]:
from datasets import load_dataset

ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")

SEQ = 128
all_ids = []
for text in ds["text"]:
    if text.strip():
        all_ids.extend(tokenizer.encode(text, add_special_tokens=False))
    if len(all_ids) > 20_000_000:
        break

n_chunks = len(all_ids) // SEQ
chunks = [torch.tensor(all_ids[i*SEQ:(i+1)*SEQ], dtype=torch.long) for i in range(n_chunks)]
loader = DataLoader(chunks, batch_size=32, shuffle=True, drop_last=True, num_workers=2)
print(f"Dataset: {n_chunks:,} chunks ({len(all_ids):,} tokens)")

## 4. Train (50k steps)

In [ ]:
STEPS = 50_000
LR = 5e-4
TEMP = 2.0
ALPHA = 0.7

opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)
WU = 1000

def lr_fn(s):
    if s < WU: return s / WU
    return 0.5 * (1 + math.cos(math.pi * (s - WU) / (STEPS - WU)))

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

student.train()
it = iter(loader)
losses = []
best = float('inf')
t0 = time.time()

print(f"Training: {STEPS:,} steps | batch=32 | lr={LR} | temp={TEMP} | alpha={ALPHA}")
print("-" * 70)

for step in range(1, STEPS + 1):
    try:
        ids = next(it).to(device)
    except StopIteration:
        it = iter(loader); ids = next(it).to(device)

    inp, labels = ids[:, :-1], ids[:, 1:]
    B, L = inp.shape

    if L < SEQ:
        s_inp = torch.cat([inp, torch.zeros(B, SEQ-L, dtype=torch.long, device=device)], 1)
    else:
        s_inp = inp[:, :SEQ]; labels = labels[:, :SEQ]; inp = inp[:, :SEQ]; L = SEQ

    sl = student(s_inp)[:, :L, :]
    with torch.no_grad():
        tl = teacher(inp).logits

    V = min(sl.size(-1), tl.size(-1))
    sl, tl = sl[:,:,:V], tl[:,:,:V]

    kl = F.kl_div(F.log_softmax(sl/TEMP, -1), F.softmax(tl/TEMP, -1), reduction="batchmean") * TEMP**2
    ce = F.cross_entropy(sl.reshape(-1, V), labels.reshape(-1))
    loss = ALPHA * kl + (1 - ALPHA) * ce

    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    opt.step(); sched.step()
    losses.append(loss.item())

    if step % 500 == 0:
        avg = sum(losses[-500:]) / 500
        el = time.time() - t0
        eta = el / step * (STEPS - step)
        tag = " *" if avg < best else ""
        if avg < best: best = avg
        print(f"Step {step:6d}/{STEPS:,} | loss={avg:.4f}{tag} | lr={sched.get_last_lr()[0]:.1e} | {el/60:.0f}m | ETA {eta/60:.0f}m")

    if step % 10_000 == 0:
        from safetensors.torch import save_file as sf
        os.makedirs("ckpt", exist_ok=True)
        st = {k: v.cpu() for k, v in student.state_dict().items() if not k.startswith(("rc","rs","mask"))}
        sf(st, f"ckpt/step_{step}.safetensors")
        print(f"  >> saved ckpt/step_{step}.safetensors")

print(f"\nDone! {(time.time()-t0)/60:.0f} min | best={best:.4f}")

## 5. Eval: measure acceptance rate

In [ ]:
student.eval()

prompts = [
    "The meaning of life is",
    "Once upon a time",
    "The capital of France is",
    "In machine learning, a neural network",
    "The president of the United States",
]

total_match, total_tok = 0, 0

for prompt in prompts:
    ids = tokenizer.encode(prompt)
    s_ctx, t_ctx = list(ids), list(ids)

    for _ in range(50):
        # Student
        padded = s_ctx[-SEQ:] + [0] * max(0, SEQ - len(s_ctx))
        inp = torch.tensor([padded[:SEQ]], device=device)
        with torch.no_grad(): logits = student(inp)
        pos = min(len(s_ctx), SEQ) - 1
        s_ctx.append(logits[0, pos].argmax().item())

        # Teacher
        with torch.no_grad(): tl = teacher(torch.tensor([t_ctx], device=device)).logits
        t_ctx.append(tl[0, -1].argmax().item())

    s_out = tokenizer.decode(s_ctx[len(ids):])
    t_out = tokenizer.decode(t_ctx[len(ids):])
    match = sum(1 for a, b in zip(s_ctx[len(ids):], t_ctx[len(ids):]) if a == b)
    total_match += match; total_tok += 50

    print(f'\n"{prompt}"')
    print(f'  Student: {s_out[:80]}')
    print(f'  Teacher: {t_out[:80]}')
    print(f'  Match: {match}/50 = {match/50:.0%}')

print(f"\n{'='*50}")
print(f"Overall greedy match: {total_match}/{total_tok} = {total_match/total_tok:.0%}")
print(f"{'='*50}")

## 6. Save

In [ ]:
from safetensors.torch import save_file

os.makedirs("draft_80m_weights", exist_ok=True)

state = {k: v.cpu() for k, v in student.state_dict().items()
         if not k.startswith(("rc", "rs", "mask"))}
save_file(state, "draft_80m_weights/draft_80m.safetensors")

with open("draft_80m_weights/config.json", "w") as f:
    json.dump(DRAFT_CONFIG, f, indent=2)

sz = os.path.getsize("draft_80m_weights/draft_80m.safetensors")
print(f"Saved: draft_80m.safetensors ({sz/1e6:.0f} MB)")
print(f"       config.json")

In [ ]:
try:
    from google.colab import files
    files.download("draft_80m_weights/draft_80m.safetensors")
    files.download("draft_80m_weights/config.json")
except:
    print("Find files in draft_80m_weights/")